# Pretraining 101 — From Scratch (Companion Notebook)

Companion to the MLWhiz post **"Pretraining 101: Data, Scale, and the Loss Function."**

Here we actually *pretrain* a tiny GPT from scratch on a real public-domain book (downloaded from Project Gutenberg). The model is deliberately small, so the whole thing runs on a **laptop CPU in a few minutes** — seconds on a GPU, which the notebook auto-detects. The point isn't a good model — it's to make the pretraining loop concrete:

1. Turn text into tokens (a char-level tokenizer)
2. Build (input → next-token) training pairs
3. Define a tiny GPT and the **cross-entropy** loss
4. Run the training loop (AdamW + warmup/cosine LR) and watch **loss / perplexity** fall
5. Sample text **before vs after** training
6. Bonus: the cross-entropy = `H(p) + KL` identity, and the `6ND` compute math

> Requirements: `torch` and `matplotlib`. If torch is missing: `pip install torch matplotlib`.


In [ ]:
# --- Setup ---
import math, torch, torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(1337)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)


## 1. The data — a real book from Project Gutenberg

A real pretraining run uses trillions of tokens. We'll use one whole public-domain book — *Alice's Adventures in Wonderland*, pulled straight from [Project Gutenberg](https://www.gutenberg.org/) (~150 KB). If you're offline, the cell falls back to a small bundled excerpt so the notebook still runs.

Want a different corpus? Point `BOOK_URL` at any Project Gutenberg plain-text link — *Pride and Prejudice* (`/cache/epub/1342/pg1342.txt`), the complete Shakespeare (`/cache/epub/100/pg100.txt`), or anything else.


In [ ]:
import urllib.request

BOOK_URL = "https://www.gutenberg.org/cache/epub/11/pg11.txt"   # Alice in Wonderland

FALLBACK_TEXT = "Alice was beginning to get very tired of sitting by her sister on the bank, and of having nothing to do: once or twice she had peeped into the book her sister was reading, but it had no pictures or conversations in it, \"and what is the use of a book,\" thought Alice, \"without pictures or conversations?\"\n\nSo she was considering in her own mind (as well as she could, for the hot day made her feel very sleepy and stupid), whether the pleasure of making a daisy-chain would be worth the trouble of getting up and picking the daisies, when suddenly a White Rabbit with pink eyes ran close by her.\n\nThere was nothing so very remarkable in that; nor did Alice think it so very much out of the way to hear the Rabbit say to itself, \"Oh dear! Oh dear! I shall be late!\" (when she thought it over afterwards, it occurred to her that she ought to have wondered at this, but at the time it all seemed quite natural); but when the Rabbit actually took a watch out of its waistcoat-pocket, and looked at it, and then hurried on, Alice started to her feet, for it flashed across her mind that she had never before seen a rabbit with either a waistcoat-pocket, or a watch to take out of it, and burning with curiosity, she ran across the field after it, and fortunately was just in time to see it pop down a large rabbit-hole under the hedge.\n\nIn another moment down went Alice after it, never once considering how in the world she was to get out again. The rabbit-hole went straight on like a tunnel for some way, and then dipped suddenly down, so suddenly that Alice had not a moment to think about stopping herself before she found herself falling down a very deep well.\n"

def load_gutenberg(url):
    """Download a Project Gutenberg book and strip its header/footer boilerplate."""
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    raw = urllib.request.urlopen(req, timeout=20).read().decode("utf-8", errors="ignore")
    raw = raw.replace("\r\n", "\n")
    start, end = raw.find("*** START OF"), raw.find("*** END OF")
    if start != -1:
        start = raw.find("\n", start) + 1
    return (raw[start:end] if (start != -1 and end != -1) else raw).strip()

try:
    text = load_gutenberg(BOOK_URL)
    print(f"Downloaded {len(text):,} characters from Project Gutenberg.")
except Exception as e:
    text = FALLBACK_TEXT
    print("Download failed (offline?); using the bundled excerpt instead. Reason:", e)

print(f"corpus length: {len(text):,} characters")
print(text[:280], "...")


## 2. Tokenizer (character-level)

The post used Byte-Pair Encoding. Here we use the simplest possible tokenizer — one token per character — so there's nothing to download or train. Same idea: **text → integers → text.**


In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

print("vocab size:", vocab_size)
print("sample encode:", encode("Alice"))
print("roundtrip:", decode(encode("Alice")))

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print("train tokens:", len(train_data), "| val tokens:", len(val_data))


## 3. Batching: build (input → next-token) pairs

This is the self-supervised trick. For a chunk of `block_size` tokens, the target is the **same chunk shifted by one**: at every position, predict the next token. No human labels — the text is its own answer key.


In [ ]:
block_size = 64   # context length
batch_size = 32

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])  # shifted by one
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("x:", xb.shape, "y:", yb.shape)
print("x[0]:", xb[0][:10].tolist())
print("y[0]:", yb[0][:10].tolist(), " <- y is x shifted left by one")


## 4. A tiny GPT

A minimal decoder-only Transformer: token + positional embeddings, a few blocks of (masked self-attention + MLP), a final layer-norm, and an **LM head** that projects back to vocabulary logits. This is the architecture from the previous post in the series; here we just need it as the thing we're training.


In [ ]:
n_embd  = 64
n_head  = 4
n_layer = 3
dropout = 0.0

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        att = (q @ k.transpose(-2, -1)) * k.shape[-1] ** -0.5
        att = att.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        att = self.drop(F.softmax(att, dim=-1))
        return att @ v

class MultiHead(nn.Module):
    def __init__(self, n_head, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_head)])
        self.proj  = nn.Linear(n_embd, n_embd)
        self.drop  = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.drop(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.GELU(),
            nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.sa  = MultiHead(n_head, n_embd // n_head)
        self.ff  = FeedForward()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))   # residual + attention
        x = x + self.ff(self.ln2(x))   # residual + MLP
        return x

class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks  = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f    = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.tok_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=idx.device))
        x = self.blocks(tok + pos)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            # next-token cross-entropy over every position
            loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, nxt], dim=1)
        return idx

model = MiniGPT().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params:,}")


## 5. The loss before training, and perplexity

At init the model is random, so it spreads probability roughly evenly over the vocabulary. Cross-entropy should be about `ln(vocab_size)`, and **perplexity = exp(loss)** should be close to the vocabulary size — "as confused as guessing uniformly among all tokens."


In [ ]:
xb, yb = get_batch("train")
_, loss0 = model(xb, yb)
print(f"initial loss        : {loss0.item():.3f}")
print(f"ln(vocab_size)      : {math.log(vocab_size):.3f}   (expected ~init loss)")
print(f"initial perplexity  : {math.exp(loss0.item()):.1f}   (vocab_size = {vocab_size})")

ctx = torch.zeros((1, 1), dtype=torch.long, device=device)
print("\n--- sample BEFORE training (gibberish) ---")
print(decode(model.generate(ctx, 200)[0].tolist()))


## 6. The training loop

Four moves per step, exactly as in the post: forward → cross-entropy loss → backprop → optimizer step. We add the standard **warmup + cosine** learning-rate schedule and track train/val loss.


In [ ]:
max_iters   = 3000
eval_every  = 300
warmup      = 150
max_lr      = 3e-3
min_lr      = 3e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr)

def lr_at(step):
    if step < warmup:
        return max_lr * (step + 1) / warmup
    r = (step - warmup) / (max_iters - warmup)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * r))

@torch.no_grad()
def eval_loss():
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = torch.zeros(20)
        for k in range(20):
            x, y = get_batch(split)
            _, l = model(x, y)
            losses[k] = l.item()
        out[split] = losses.mean().item()
    model.train()
    return out

history = []
for step in range(max_iters + 1):
    for g in optimizer.param_groups:      # apply LR schedule
        g["lr"] = lr_at(step)
    xb, yb = get_batch("train")           # 1. batch
    _, loss = model(xb, yb)               # 2. forward + cross-entropy
    optimizer.zero_grad(set_to_none=True)
    loss.backward()                       # 3. backprop
    optimizer.step()                      # 4. update
    if step % eval_every == 0:
        m = eval_loss()
        history.append((step, m["train"], m["val"]))
        print(f"step {step:4d} | train {m['train']:.3f} | val {m['val']:.3f} "
              f"| ppl {math.exp(m['val']):.1f} | lr {lr_at(step):.4f}")


## 7. The loss curve

The smooth downward sweep from the post — steep at first, then a slow grind. Watch perplexity collapse from ~vocab-size toward single digits.


In [ ]:
steps   = [h[0] for h in history]
tr      = [h[1] for h in history]
va      = [h[2] for h in history]

plt.figure(figsize=(7, 4))
plt.plot(steps, tr, label="train loss")
plt.plot(steps, va, label="val loss")
plt.xlabel("training step")
plt.ylabel("cross-entropy loss")
plt.title("Pretraining loss curve")
plt.legend(); plt.grid(alpha=0.3)
plt.show()
print(f"final perplexity (val): {math.exp(va[-1]):.2f}")


## 8. Sample after training

Same model, after pretraining. With this little data it will mostly memorize the source text, but you'll see it has gone from random characters to real words and Alice-shaped sentences — that's the loss function doing its job.


In [ ]:
ctx = torch.zeros((1, 1), dtype=torch.long, device=device)
print("--- sample AFTER training ---")
print(decode(model.generate(ctx, 400, temperature=0.8)[0].tolist()))


## 9. Bonus — cross-entropy is KL divergence in disguise

The identity from the post: `cross_entropy(p, q) = H(p) + KL(p || q)`. `H(p)` is the entropy of the data (a fixed floor); the only part training can shrink is the KL divergence.


In [ ]:
p = torch.tensor([0.6, 0.2, 0.1, 0.07, 0.03])
q = torch.tensor([0.3, 0.3, 0.2, 0.10, 0.10])

H_p = -(p * p.log()).sum()
ce  = -(p * q.log()).sum()
kl  =  (p * (p / q).log()).sum()

print(f"H(p)          = {H_p:.3f}")
print(f"cross-entropy = {ce:.3f}")
print(f"KL(p||q)      = {kl:.3f}")
print(f"H(p) + KL     = {H_p + kl:.3f}   <- equals cross-entropy")


## 10. Bonus — the compute budget (`C ≈ 6ND`) and Chinchilla

The back-of-envelope math behind real training runs: compute ≈ 6 × parameters × tokens, and the Chinchilla rule of thumb of ~20 tokens per parameter.


In [ ]:
def training_flops(num_params, num_tokens):
    return 6 * num_params * num_tokens

def chinchilla_optimal_tokens(num_params):
    return 20 * num_params

print(f"This notebook's model : {n_params:,} params")
print(f"GPT-3   (175B, 300B tok)  FLOPs : {training_flops(175e9, 300e9):.2e}")
print(f"GPT-3 'should' have       tokens: {chinchilla_optimal_tokens(175e9):.2e}  (it trained on ~3e11)")
print(f"Llama 3 8B (15T tok)      FLOPs : {training_flops(8e9, 15e12):.2e}")
print(f"Llama 3 8B tokens/param          : {15e12 / 8e9:.0f} : 1  (~100x past Chinchilla)")


## Wrap-up

That's pretraining, end to end and from scratch: tokens in, next-token cross-entropy, backprop, repeat — until a pile of random weights turns into something that completes text.

Things to try:
- Bump `max_iters`, `n_layer`, or `n_embd` and watch the loss curve change.
- Swap in your own `text` (a few KB of any plain text).
- Lower the sampling `temperature` toward 0 for more deterministic output.

Back to the post: **Pretraining 101: Data, Scale, and the Loss Function.**
